# OEPolars - Advanced Features

This notebook demonstrates advanced OEPolars features including file I/O, design units, data quality filtering, lazy evaluation, and performance optimization.

In [1]:
import oepolars as oep
import polars as pl
from openeye import oechem, oemolprop
from pathlib import Path
import tempfile
import time

## File I/O

### Reading Different File Formats

OEPolars supports reading various molecular file formats with customizable column names:

```python
# SD files - SD data becomes DataFrame columns
df = oep.read_sdf(
    "molecules.sdf",
    molecule_column="Mol",      # Name for the molecule column (default: "Molecule")
    title_column="Name",         # Name for the title column (default: "Title")
    usecols=["Activity", "MW"],  # Only read specific SD tags
    numeric_columns=["Activity", "MW"]   # Convert these columns to numeric types
)

# OEB binary files (supports .oeb and .oeb.gz)
df = oep.read_oeb(
    "molecules.oeb.gz",
    molecule_column="Mol",
    title_column="Name"
)

# SMILES files
df = oep.read_smi(
    "molecules.smi",
    molecule_column="Mol",
    title_column="Name"
)

# CSV with SMILES column
df = oep.read_molecule_csv(
    "data.csv",
    smiles_column="SMILES"  # Column to convert to molecules
)

# OERecord databases
df = oep.read_oedb("records.oedb")

# Design unit files
df = oep.read_oedu("complexes.oedu")
```

### Lazy Reading with Scanners

For large files, use `scan_*` functions to create LazyFrames:

```python
# Lazy reading - returns LazyFrame for query optimization
lf = oep.scan_sdf("large_file.sdf")
lf = oep.scan_oeb("large_file.oeb")
lf = oep.scan_smi("large_file.smi")
lf = oep.scan_oedb("large_file.oedb")
lf = oep.scan_oedu("large_file.oedu")
lf = oep.scan_molecule_csv("large_file.csv", smiles_column="SMILES")
lf = oep.scan_parquet("large_file.parquet", molecule_columns=["Mol"])

# Apply filters before collecting
result = lf.filter(pl.col("MW") > 300).collect()
```

### Writing Data

Export molecular data to various formats using the `.chem` accessor:

In [2]:
# Create sample data
df = pl.DataFrame({
    "SMILES": ["CC(=O)Oc1ccccc1C(=O)O", "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "CC(=O)Nc1ccc(cc1)O"],
    "Name": ["Aspirin", "Ibuprofen", "Acetaminophen"],
    "Activity": [7.5, 6.8, 5.2]
})
df = df.chem.as_molecule("SMILES")

print(f"Created DataFrame with {len(df)} molecules")
df

Created DataFrame with 3 molecules


SMILES,Name,Activity
ext[molecule],str,f64
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e67d170> >,"""Aspirin""",7.5
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10b0dbe40> >,"""Ibuprofen""",6.8
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10b0dad90> >,"""Acetaminophen""",5.2


In [3]:
# Write to various formats
with tempfile.TemporaryDirectory() as tmpdir:
    output_dir = Path(tmpdir)
    
    # Write to SDF (columns become SD tags)
    df.chem.write_sdf(
        output_dir / "output.sdf",
        molecule_column="SMILES",
        title_column="Name",
        sd_columns=["Activity"]  # Include these columns as SD tags
    )
    print("Wrote SDF file")
    
    # Write to SMILES file
    df.chem.write_smi(
        output_dir / "output.smi",
        molecule_column="SMILES",
        title_column="Name"
    )
    print("Wrote SMILES file")
    
    # Write to OEB binary file
    df.chem.write_oeb(
        output_dir / "output.oeb",
        molecule_column="SMILES",
        title_column="Name",
        sd_columns=["Activity"]
    )
    print("Wrote OEB file")
    
    # Write to molecule CSV (molecules as SMILES strings)
    df.chem.write_molecule_csv(
        output_dir / "output.csv",
        molecule_column="SMILES",
        smiles_column="smiles_string"
    )
    print("Wrote molecule CSV file")
    
    # Write to OEDB database
    df.chem.write_oedb(
        output_dir / "output.oedb",
        molecule_column="SMILES",
        title_column="Name",
        sd_columns=["Activity"]
    )
    print("Wrote OEDB file")
    
    # Write to Parquet with molecule serialization
    df.chem.write_parquet(
        output_dir / "output.parquet"
    )
    print("Wrote Parquet file")
    
    # List created files
    print(f"\nFiles created in {output_dir}:")
    for f in sorted(output_dir.iterdir()):
        print(f"  - {f.name} ({f.stat().st_size} bytes)")

Wrote SDF file
Wrote SMILES file
Wrote OEB file
Wrote molecule CSV file
Wrote OEDB file
Wrote Parquet file

Files created in /var/folders/dk/nb_vhq1s6xggs22mtl34dc2m0000gp/T/tmp8jb8bnbf:
  - output.csv (140 bytes)
  - output.oeb (770 bytes)
  - output.oedb (770 bytes)
  - output.parquet (3286 bytes)
  - output.sdf (3902 bytes)
  - output.smi (100 bytes)


## Working with Design Units

Design units are OpenEye's container for protein-ligand complexes:

```python
# Read design unit file
df_du = oep.read_oedu(
    "complexes.oedu",
    design_unit_column="DU",  # Name for design unit column (default: "DesignUnit")
    title_column="Title"       # Name for title column (default: "Title")
)

# Extract components using .chem accessor
df_du = df_du.with_columns([
    df_du["DU"].chem.get_ligands().alias("Ligand"),
    df_du["DU"].chem.get_proteins().alias("Protein")
])

# Calculate ligand properties
df_du = df_du.with_columns(
    pl.col("Ligand").map_elements(
        lambda mol: oechem.OECalculateMolecularWeight(mol) if mol else None,
        return_dtype=pl.Float64
    ).alias("LigandMW")
)

# Write design units back to file
df_du.chem.write_oedu("output.oedu", design_unit_column="DU")
```

## Data Quality and Filtering

### Filtering Invalid Molecules

OEPolars provides built-in methods for checking and filtering molecule validity:

In [4]:
# Create data with some invalid molecules
df_test = pl.DataFrame({
    "SMILES": ["CCO", "invalid", "c1ccccc1", "not_a_molecule", "CC(=O)O"],
    "Name": ["Ethanol", "Invalid1", "Benzene", "Invalid2", "Acetic Acid"]
})
df_test = df_test.chem.as_molecule("SMILES")

print(f"Created DataFrame with {len(df_test)} entries")

Created DataFrame with 5 entries


In [5]:
# Check validity using .chem.is_valid()
validity = df_test["SMILES"].chem.is_valid()

print("Molecule validity check:")
for name, is_valid in zip(df_test["Name"].to_list(), validity.to_list()):
    status = "Valid" if is_valid else "INVALID"
    print(f"  {name}: {status}")

print(f"\nValid molecules: {validity.sum()}")
print(f"Invalid molecules: {(~validity).sum()}")

Molecule validity check:
  Ethanol: Valid
  Invalid1: INVALID
  Benzene: Valid
  Invalid2: INVALID
  Acetic Acid: Valid

Valid molecules: 3
Invalid molecules: 2


In [6]:
# Filter to keep only valid molecules using .chem.filter_valid()
df_valid = df_test.chem.filter_valid("SMILES")

print(f"Original: {len(df_test)} rows")
print(f"After filtering: {len(df_valid)} rows")
print("\nValid molecules:")
df_valid

Original: 5 rows
After filtering: 3 rows

Valid molecules:


SMILES,Name
ext[molecule],str
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1195325b0> >,"""Ethanol"""
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e6db2d0> >,"""Benzene"""
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10b0d91a0> >,"""Acetic Acid"""


In [7]:
# You can also filter multiple columns at once
# df_valid = df.chem.filter_valid(["SMILES", "Product"])  # Filter rows where both are valid

# Or use is_valid() for custom filtering
df_with_validity = df_test.with_columns(
    df_test["SMILES"].chem.is_valid().alias("IsValid")
)
df_with_validity

SMILES,Name,IsValid
ext[molecule],str,bool
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1195325b0> >,"""Ethanol""",true
null,"""Invalid1""",false
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e6db2d0> >,"""Benzene""",true
null,"""Invalid2""",false
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10b0d91a0> >,"""Acetic Acid""",true


### Property-Based Filtering

In [8]:
# Create a larger dataset for filtering examples
drug_data = pl.DataFrame({
    "SMILES": [
        "CC(=O)Oc1ccccc1C(=O)O",
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
        "CC(=O)Nc1ccc(cc1)O",
        "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
        "CN1c2ccc(cc2C(=NCC1=O)c3ccccc3)Cl"
    ],
    "Name": ["Aspirin", "Ibuprofen", "Acetaminophen", "Caffeine", "Diazepam"]
})
df_drugs = drug_data.chem.as_molecule("SMILES")

# Calculate properties
df_drugs = df_drugs.with_columns([
    pl.col("SMILES").map_elements(
        lambda mol: oechem.OECalculateMolecularWeight(mol) if mol else None,
        return_dtype=pl.Float64
    ).alias("MW"),
    pl.col("SMILES").map_elements(
        lambda mol: oemolprop.OEGetXLogP(mol) if mol else None,
        return_dtype=pl.Float64
    ).alias("XLogP"),
    pl.col("SMILES").map_elements(
        lambda mol: oemolprop.OEGetHBondDonorCount(mol) if mol else None,
        return_dtype=pl.Int64
    ).alias("HBD"),
    pl.col("SMILES").map_elements(
        lambda mol: oemolprop.OEGetHBondAcceptorCount(mol) if mol else None,
        return_dtype=pl.Int64
    ).alias("HBA")
])

df_drugs.select(["Name", "MW", "XLogP", "HBD", "HBA"])

Name,MW,XLogP,HBD,HBA
str,f64,f64,i64,i64
"""Aspirin""",180.15742,0.783,1,2
"""Ibuprofen""",206.28082,2.878,1,1
"""Acetaminophen""",151.16256,0.432,2,1
"""Caffeine""",194.1906,-0.607,0,3
"""Diazepam""",284.74022,3.214,0,2


In [9]:
# Apply Lipinski's Rule of Five filter using Polars expressions
df_drugs = df_drugs.with_columns(
    ((pl.col("MW") <= 500) & 
     (pl.col("XLogP") <= 5) & 
     (pl.col("HBD") <= 5) & 
     (pl.col("HBA") <= 10)).alias("PassesRO5")
)

print(f"Molecules passing Lipinski's Rule of Five: {df_drugs['PassesRO5'].sum()}")
print(f"Molecules failing: {(~df_drugs['PassesRO5']).sum()}")

# Filter to RO5-compliant molecules
df_ro5 = df_drugs.filter(pl.col("PassesRO5"))
df_ro5.select(["Name", "MW", "XLogP", "HBD", "HBA"])

Molecules passing Lipinski's Rule of Five: 5
Molecules failing: 0


Name,MW,XLogP,HBD,HBA
str,f64,f64,i64,i64
"""Aspirin""",180.15742,0.783,1,2
"""Ibuprofen""",206.28082,2.878,1,1
"""Acetaminophen""",151.16256,0.432,2,1
"""Caffeine""",194.1906,-0.607,0,3
"""Diazepam""",284.74022,3.214,0,2


## Lazy Evaluation

OEPolars supports Polars LazyFrames for query optimization. This is especially useful for large datasets.

In [10]:
# Create a LazyFrame from a DataFrame
lf = df_drugs.lazy()

# Check LazyFrame capabilities
print(f"Has molecule columns: {lf.chem.has_molecule_columns()}")
print(f"Molecule columns: {lf.chem.molecule_columns()}")

Has molecule columns: True
Molecule columns: ['SMILES']


In [11]:
# Apply filters on non-molecule columns before collecting
result = (
    lf
    .filter(pl.col("MW") > 180)
    .filter(pl.col("XLogP") < 3)
    .select(["Name", "MW", "XLogP"])
    .collect()
)

print("Filtered results (MW > 180 and XLogP < 3):")
result

Filtered results (MW > 180 and XLogP < 3):


Name,MW,XLogP
str,f64,f64
"""Aspirin""",180.15742,0.783
"""Ibuprofen""",206.28082,2.878
"""Caffeine""",194.1906,-0.607


In [12]:
# For molecule operations, collect first, then use .chem accessor
collected_df = lf.collect()
matches = collected_df["SMILES"].chem.subsearch("c1ccccc1")  # Find aromatic rings

print("Molecules with aromatic rings:")
collected_df.filter(matches).select(["Name", "MW"])

Molecules with aromatic rings:


Name,MW
str,f64
"""Aspirin""",180.15742
"""Ibuprofen""",206.28082
"""Acetaminophen""",151.16256
"""Diazepam""",284.74022


## Performance Optimization

### Bulk Operations

In [13]:
# Create a larger dataset for performance testing
large_smiles = [
    "CC(=O)Oc1ccccc1C(=O)O",
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "CC(=O)Nc1ccc(cc1)O",
    "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
    "c1ccccc1",
] * 200  # 1000 molecules

df_large = pl.DataFrame({"SMILES_str": large_smiles})
df_large = df_large.chem.as_molecule("SMILES_str")

print(f"Created DataFrame with {len(df_large)} molecules")

Created DataFrame with 1000 molecules


In [14]:
# Time bulk operations
start = time.time()
atom_counts = df_large.with_columns(
    pl.col("SMILES_str").map_elements(
        lambda mol: mol.NumAtoms() if mol else 0,
        return_dtype=pl.Int64
    ).alias("atom_count")
)
elapsed = time.time() - start

print(f"Calculated atom counts for {len(df_large)} molecules")
print(f"Total time: {elapsed:.4f} seconds")
print(f"Average: {elapsed/len(df_large)*1000:.3f} ms per molecule")

Calculated atom counts for 1000 molecules
Total time: 0.0010 seconds
Average: 0.001 ms per molecule


In [15]:
# Using built-in validity check
start = time.time()
validity = df_large["SMILES_str"].chem.is_valid()
elapsed = time.time() - start

print(f"Checked validity for {len(df_large)} molecules")
print(f"Total time: {elapsed:.4f} seconds")
print(f"Valid molecules: {validity.sum()}")

Checked validity for 1000 molecules
Total time: 0.0002 seconds
Valid molecules: 1000


In [16]:
# Substructure search performance
start = time.time()
matches = df_large["SMILES_str"].chem.subsearch("c1ccccc1")
elapsed = time.time() - start

print(f"Substructure search on {len(df_large)} molecules")
print(f"Total time: {elapsed:.4f} seconds")
print(f"Matches: {matches.sum()}")

Substructure search on 1000 molecules
Total time: 0.0016 seconds
Matches: 800


## Advanced Molecular Operations

### Conformer Generation

In [17]:
from openeye import oeomega

def generate_conformers(mol, max_confs=10):
    """Generate 3D conformers for a molecule."""
    if mol is None:
        return None
    
    omega = oeomega.OEOmega()
    omega.SetMaxConfs(max_confs)
    omega.SetIncludeInput(False)
    omega.SetStrictStereo(False)
    
    mol_copy = oechem.OEMol(mol)
    if omega(mol_copy):
        return mol_copy
    return None

# Generate conformers for first molecule
sample_mol = df_drugs["SMILES"][0]
mol_with_confs = generate_conformers(sample_mol, max_confs=5)

if mol_with_confs:
    print(f"Original molecule: {df_drugs['Name'][0]}")
    print(f"Generated {mol_with_confs.NumConfs()} conformers")

Original molecule: Aspirin
Generated 5 conformers


### Molecular Fingerprints for Similarity

In [18]:
from openeye import oegraphsim

def calc_fingerprint(mol):
    """Calculate circular fingerprint."""
    if mol is None:
        return None
    fp = oegraphsim.OEFingerPrint()
    oegraphsim.OEMakeFP(fp, mol, oegraphsim.OEFPType_Circular)
    return fp

def calc_tanimoto(fp1, fp2):
    """Calculate Tanimoto similarity between fingerprints."""
    if fp1 is None or fp2 is None:
        return 0.0
    return oegraphsim.OETanimoto(fp1, fp2)

# Calculate fingerprints (store as Object column)
fps = [calc_fingerprint(mol) for mol in df_drugs["SMILES"].to_list()]
df_drugs = df_drugs.with_columns(
    pl.Series("Fingerprint", fps, dtype=pl.Object)
)

# Calculate pairwise similarity to first molecule
reference_fp = df_drugs["Fingerprint"][0]
similarities = [calc_tanimoto(reference_fp, fp) for fp in df_drugs["Fingerprint"].to_list()]
df_drugs = df_drugs.with_columns(
    pl.Series("Similarity", similarities)
)

print(f"Similarity to {df_drugs['Name'][0]}:")
df_drugs.select(["Name", "Similarity"]).sort("Similarity", descending=True)

Similarity to Aspirin:


Name,Similarity
str,f64
"""Aspirin""",1.0
"""Acetaminophen""",0.183673
"""Ibuprofen""",0.147541
"""Diazepam""",0.120482
"""Caffeine""",0.064516


## Integration with Machine Learning

Prepare molecular descriptors for ML workflows:

In [19]:
# Calculate comprehensive descriptors
df_ml = df_drugs.clone()

# Helper to safely get ring system count
def get_ring_count(mol):
    if mol is None:
        return 0
    result = oechem.OEDetermineRingSystems(mol)
    return result[0] if isinstance(result, tuple) else result

# Physicochemical properties
df_ml = df_ml.with_columns([
    pl.col("SMILES").map_elements(
        lambda mol: oemolprop.OEGet2dPSA(mol) if mol else 0,
        return_dtype=pl.Float64
    ).alias("TPSA"),
    pl.col("SMILES").map_elements(
        lambda mol: oechem.OECount(mol, oechem.OEIsRotor()) if mol else 0,
        return_dtype=pl.Int64
    ).alias("RotBonds"),
    pl.col("SMILES").map_elements(
        get_ring_count,
        return_dtype=pl.Int64
    ).alias("RingCount"),
    pl.col("SMILES").map_elements(
        lambda mol: oechem.OECount(mol, oechem.OEIsHeavy()) if mol else 0,
        return_dtype=pl.Int64
    ).alias("HeavyAtoms")
])

# Define feature columns
feature_columns = ["MW", "XLogP", "HBD", "HBA", "TPSA", "RotBonds", "RingCount", "HeavyAtoms"]

# Create feature matrix
X = df_ml.select(feature_columns).to_numpy()

print(f"Feature matrix shape: {X.shape}")
print(f"\nFeature statistics:")
df_ml.select(feature_columns).describe()

Feature matrix shape: (5, 8)

Feature statistics:


statistic,MW,XLogP,HBD,HBA,TPSA,RotBonds,RingCount,HeavyAtoms
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",203.306324,1.34,0.8,1.8,48.943999,2.0,1.2,14.6
"""std""",49.945639,1.643384,0.83666,0.83666,13.97466,1.581139,0.447214,3.361547
"""min""",151.16256,-0.607,0.0,1.0,32.669998,0.0,1.0,11.0
"""25%""",180.15742,0.432,0.0,1.0,37.299999,1.0,1.0,13.0
"""50%""",194.1906,0.783,1.0,2.0,49.329998,2.0,1.0,14.0
"""75%""",206.28082,2.878,1.0,2.0,61.82,3.0,1.0,15.0
"""max""",284.74022,3.214,2.0,3.0,63.599998,4.0,2.0,20.0


In [20]:
# Example: Simple clustering based on descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Cluster molecules
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_ml = df_ml.with_columns(
    pl.Series("Cluster", clusters)
)

print("Clustering results:")
df_ml.select(["Name", "Cluster"]).sort("Cluster")

Clustering results:


Name,Cluster
str,i32
"""Aspirin""",0
"""Ibuprofen""",0
"""Acetaminophen""",0
"""Caffeine""",0
"""Diazepam""",1


## Parquet Serialization

OEPolars supports efficient storage of molecular data in Parquet format. Molecules are serialized to binary OEB format before storage, allowing full molecule reconstruction on read.

In [21]:
with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "molecules.parquet"
    
    # Save DataFrame with molecules
    df_save = df_drugs.select(["SMILES", "Name", "MW", "XLogP"])
    
    # Write to Parquet - molecules are automatically serialized
    df_save.chem.write_parquet(parquet_path)
    print(f"Saved {len(df_save)} molecules to Parquet ({parquet_path.stat().st_size} bytes)")
    
    # Read back - specify which columns contain molecules
    df_loaded = oep.read_parquet(parquet_path, molecule_columns=["SMILES"])
    
    print(f"\nLoaded {len(df_loaded)} molecules")
    print(f"SMILES column dtype: {df_loaded.schema['SMILES']}")
    
    # Verify molecules are intact
    original_smiles = df_save["SMILES"].chem.to_smiles().to_list()
    loaded_smiles = df_loaded["SMILES"].chem.to_smiles().to_list()
    
    print(f"\nRoundtrip verification:")
    for name, orig, loaded in zip(df_loaded["Name"].to_list(), original_smiles, loaded_smiles):
        match = "MATCH" if orig == loaded else "MISMATCH"
        print(f"  {name}: {match}")

Saved 5 molecules to Parquet (3868 bytes)

Loaded 5 molecules
SMILES column dtype: MoleculeType('molecule', Object)

Roundtrip verification:
  Aspirin: MATCH
  Ibuprofen: MATCH
  Acetaminophen: MATCH
  Caffeine: MATCH
  Diazepam: MATCH


In [22]:
# Lazy reading from Parquet
with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "molecules.parquet"
    df_save.chem.write_parquet(parquet_path)
    
    # Use scan_parquet for lazy reading
    lf = oep.scan_parquet(parquet_path, molecule_columns=["SMILES"])
    
    # Apply filters before collecting
    result = (
        lf
        .filter(pl.col("MW") > 180)
        .collect()
    )
    
    print(f"Filtered result: {len(result)} molecules with MW > 180")
    result

Filtered result: 4 molecules with MW > 180
